In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# TEM関数でキック・イジング・モデルをシミュレートする

import TutorialFeedback from '@site/src/components/TutorialFeedback';



AlgorithmiqのTensor-network Error Mitigation（TEM）手法は、古典的な後処理段階で完全に誤り軽減を行うハイブリッド量子古典アルゴリズムです。TEMを使用すると、ユーザーは量子ハードウェアで避けられないノイズによる誤りを軽減した観測量の期待値を、精度とコスト効率を向上させて計算できます。これにより、量子研究者や産業実践者にとって非常に魅力的な選択肢となっています。

このチュートリアルでは、TEMが誤り軽減なしにはアクセスできず、PECやZNEなどの他の誤り軽減手法を使用した場合に大幅に多くの量子リソースが必要となる量子システムのダイナミクスについて、意味のある結果を得られることを示します。

*使用量の見積もり：このノートブックはHeron r3デバイスで約10 QPU分を使用します。ランタイムは選択したデバイスによって大幅に異なる場合があります。セクションごとの使用量の見積もりは以下に記載しています。*
## TEMファンクションで誤り軽減された多体物理実験を実行する
このチュートリアルは次の参考文献に基づいています：[L. E. Fischer et al., Nat. Phys. (2026)](https://www.nature.com/articles/s41567-025-03144-9)。この参考文献では、最大91量子ビットの量子ハードウェア上での実際のシミュレーションについて論じています。このチュートリアルでは、より小さな回路サイズで同様のシミュレーションを再現します。

キック・イジング・モデルは通常のイジング・モデルに対応します：

$$ \hat{H}_{\text{I}} = J \sum_{n=0}^{N-2} \hat{Z}_n \hat{Z}_{n+1} + h \sum_{n=0}^{N-1} \hat{Z}_n $$

にトランスバーキックが加えられます：

$$ \hat{H}_{K} = b \sum_{n=0}^{N-1} \hat{X}_n $$

目標は、トランスバーキック・イジング・ハミルトニアンの下での状態のダイナミクスをシミュレートすることです。その時間発展はFloquet単体 $\hat{U}_{\text{KI}} = e^{-i \hat{H}_K} e^{-i \hat{H}_I}$ で実装できます。進化させる初期状態は、最初の量子ビットが $|+\rangle$ の状態にあり、他の量子ビットはペアになってベル状態 $(|00\rangle + |11\rangle)/\sqrt{2}$ に設定されています。

観測したい量は相関関数です。[参考論文](https://www.nature.com/articles/s41567-025-03144-9)では、この量が $n^{th}$ 量子ビットの $\hat{X}$ パウリ演算子として書き直せることを説明しています。
いくつかの物理的時間ステップ $t$ の後、パウリ演算子 $\hat{X}_{n=t}$ の値を計算します。
システムのパラメータによって、この観測量の値は厳密に計算できるか、または近似手法でのみシミュレートできます。具体的には、$|J|=|b|=\pi/4$ のとき $[\cos(2h)]^t$ に等しくなります。これはこのチュートリアルの結果をベンチマークするために使用する値です。さらに、与えられた時間ステップ $t$ において、$\langle\hat{X}_{n\neq t}\rangle$ はゼロです。これらの値を得るための詳細、およびこれらのパラメータ以外での近似古典シミュレーション結果との比較については、[L. E. Fischer et al., Nat. Phys. (2026)](https://www.nature.com/articles/s41567-025-03144-9)を参照してください。

TEMはまず、回路内の2量子ビットゲートの各ユニークな層のノイズを特性評価し、読み出し誤りを特性評価することから始まります。次に、回路が量子機械で実行されます。最後に、テンソルネットワーク誤り軽減がIBM Cloud&reg;上の古典的なリソースで実行され、軽減された値が返されます。この例では、回路に特性評価する2つのユニークな層があります。
## セットアップ
前提条件として、必要な依存関係がインストールされていることを確認してください。

In [ ]:
%pip install numpy matplotlib qiskit qiskit-ibm-catalog qiskit-ibm-runtime pylatexenc qiskit_qasm3_import

In [1]:
import os
from matplotlib import pyplot as plt
import numpy as np

from qiskit.quantum_info import SparsePauliOp
from qiskit.qasm3 import load

from qiskit_ibm_catalog import QiskitFunctionsCatalog

## TEMによる誤り軽減
ここでは、上記のキック・イジング・モデルを実装する回路を提供します。回路は以下のように準備されます。まず、状態準備フェーズがあります。最初の量子ビットは $|+\rangle$ の状態にあり、他の量子ビットはベル対 $(|00\rangle + |11\rangle)/\sqrt{2}$ の状態にあります。これに続いて、単体発展 $\hat{U}_{\text{KI}}$ を実装するブリックワーク構造があります。物理的時間ステップの数は $t/2$ 回路層に対応します。
次のコードは、このチュートリアルに必要な2つのQASMファイルをダウンロードします。

In [ ]:
# Download required QASM files
import urllib

urllib.request.urlretrieve(
    "https://ibm.box.com/shared/static/swy5jtq309b0xpzluzlmsmj908yphes8.qasm",
    "ki_30q.qasm",
)
urllib.request.urlretrieve(
    "https://ibm.box.com/shared/static/et3gkodonw6gsp2trs43lzaozrdtiu7s.qasm",
    "ki_12q.qasm",
)

12量子ビットと6タイムステップを持つ小さなバージョンの回路を視覚化できます：

In [3]:
# Parameters of the kicked Ising model
h = 0.0
num_qubits = 12
t_steps = 6

# Load the circuit for the kicked Ising model
small_circuit = load("ki_12q.qasm")

# Draw the circuit
small_circuit.draw("mpl", scale=0.25, fold=-1)

<Image src="../docs/images/tutorials/simulate-kicked-ising-tem/extracted-outputs/381a4e25-bc9c-47d0-b9f1-172eb5516484-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/simulate-kicked-ising-tem/extracted-outputs/381a4e25-bc9c-47d0-b9f1-172eb5516484-0.avif)

次に、観測量 $\hat{X}_{n=t}$ を構築します。これはQiskitが使用する順序に合わせたシンプルなパウリ文字列として構築されます：

In [3]:
def xt_observable(n_qubits, t_steps):
    pauli_str = "".join(["I" * t_steps, "X", "I" * (n_qubits - t_steps - 1)])
    pauli_str = pauli_str[::-1]  # Reverse the string to match qiskit order
    return SparsePauliOp(data=pauli_str, coeffs=1.0)

12量子ビットの小さな例では、観測量は次のようになります：

In [4]:
# Build the observable for the kicked Ising model
small_observable = xt_observable(n_qubits=12, t_steps=6)
print(small_observable)

SparsePauliOp(['IIIIIXIIIIII'],
              coeffs=[1.+0.j])


Qiskit Functions use PUBs as the way to collect the inputs. In our case, let's consider a single circuit and observable as our PUB:

In [5]:
# Collect the input PUBs, in this case composed of a single circuit and observable
pubs = [(small_circuit, [small_observable])]

Qiskit FunctionsはPUBを入力を収集する方法として使用します。この場合、単一の回路と観測量をPUBとして考えましょう：

In [6]:
# Set IBM Quantum credentials and backend configuration
personal_token = os.environ.get(
    "QISKIT_IBM_TOKEN", "<API-KEY>"
)  # Replace with your personal token or set the environment variable
channel = "ibm_quantum_platform"
crn = "your_crn"  # Replace with the Cloud Resource Name (CRN)

# Select the QPU backend
backend_name = "ibm_qpu_name"  # Replace with your desired backend's name

次に、TEM関数にアクセスします。まず、IBM Cloudへの必要な認証を設定し、利用可能なデバイスからバックエンドを選択します。トークン、利用可能なバックエンド、対応するクラウドリソース名（CRN）は、[IBM Quantum Platformダッシュボード](https://quantum.cloud.ibm.com/)でアカウントにログインすることで取得できます。

In [8]:
# Load the TEM function from the Qiskit Functions Catalog
catalog = QiskitFunctionsCatalog(
    channel=channel,
    token=personal_token,
    instance=crn,
)
tem = catalog.load("algorithmiq/tem")

[Qiskit Functions Catalog](https://quantum.cloud.ibm.com/functions)からTEM関数を読み込みます：

In [9]:
tem_job = tem.run(pubs=pubs, backend_name=backend_name)

TEMが提供する誤り軽減でキック・イジング回路の実験を実行できます。デフォルト設定を使用すると、TEMはQPUによって異なりますが、約2.5分のQPU実行時間を期待して簡単に実行できます：

In [10]:
print(tem_job.status())

QUEUED


デフォルトオプションでは、TEM関数は量子コンピューター上で3つのジョブを実行します：ノイズ学習、読み出し軽減、回路サンプリングです。これらのそれぞれで使用されるショット数は、関数に渡されるオプションで変更できます。デフォルトでは、これらのパラメータは軽減された期待値の精度0.05を達成するように設定されています。
ジョブのステータスは[IBM Quantum Platformダッシュボード](https://quantum.cloud.ibm.com/)で確認するか、以下で確認できます：

In [11]:
# Get the results of the TEM job
tem_results = tem_job.result()[
    0
]  # Get the first and only result from the job
tem_evs = tem_results.data.evs[0]
tem_std = tem_results.data.stds[0]

print(f"TEM Result: {tem_evs:.3f} ± {tem_std:.3f}")

TEM Result: 1.031 ± 0.046


We can also check how much quantum runtime was used for each call at [IBM Quantum Platform](https://quantum.cloud.ibm.com), or by inspecting the result metadata from the Python code.

In [12]:
# Get the TEM job runtime
tem_runtime = tem_job.result().metadata["resource_usage"][
    "RUNNING: EXECUTING_QPU"
]["QPU_TIME"]

print(f"TEM Runtime: {tem_runtime} seconds")

TEM Runtime: 155.0 seconds


ステータスが `DONE` になったら、未加工の結果と軽減された結果を確認できます。以下で定義される `tem_evs` は要求された観測量の期待値（この場合は1つの観測量 $\langle \hat X_{n=t}\rangle$ のみ）であり、`tem_std` は対応する標準偏差です。

In [28]:
options = {
    "default_shots": 10_000,
    "tem_max_bond_dimension": 512,
    "tem_compression_cutoff": 1e-16,
    "compute_shadows_bias_from_observable": True,  # This option helps optimizing the measurement
    # stage since the observable is strongly biased toward the X operator for all the qubits.
    "private": False,  # set to True to keep experiment results private, recommended for confidential circuits
}

Custom options for the noise learner can also be passed. They follow the definitions used in the Qiskit Runtime [`NoiseLearnerOptions`](/docs/api/qiskit-ibm-runtime/options-noise-learner-options):

In [29]:
nl_options = {
    "num_randomizations": 32,
    "max_layers_to_learn": 2,
    "shots_per_randomization": 128,
    "layer_pair_depths": [0, 1, 2, 4, 16, 32],
}

# add noise learning options to the overall options
options |= nl_options

また、[IBM Quantum Platform](https://quantum.cloud.ibm.com)の各呼び出しで使用された量子ランタイムを確認するか、Pythonコードから結果メタデータを検査することもできます。

In [30]:
tem_job_custom = tem.run(
    pubs=pubs, backend_name=backend_name, options=options
)

If the job is not set as private, we can recover the result at a later point. To do so, save the job ID printed here and use `tem_job_custom = catalog.get_job_by_id("your-job-id")`.

In [31]:
job_id = tem_job_custom.job_id
print(f"Job ID: {job_id}")

Job ID: 1ba10094-a541-457a-9287-dbd49306d12d


In [33]:
results_custom = tem_job_custom.result()
tem_evs = results_custom[0].data.evs[0]
tem_std = results_custom[0].data.stds[0]

print(f"TEM Result: {tem_evs:.3f} ± {tem_std:.3f}")

TEM Result: 0.956 ± 0.018


ノイズ学習器のカスタムオプションも渡すことができます。これらはQiskit Runtimeの[`NoiseLearnerOptions`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-noise-learner-options)で使用される定義に従います：

In [34]:
metadata_custom = results_custom[0].metadata

unmitigated_evs = metadata_custom["evs_non_mitigated"][0]
unmitigated_stds = metadata_custom["stds_non_mitigated"][0]
print(f"Unmitigated Result: {unmitigated_evs:.3f} ± {unmitigated_stds:.3f}")

# Exact result for the kicked Ising model from the reference paper
exact_evs = np.cos(2 * h) ** t_steps
print("Exact Result:", exact_evs)

Unmitigated Result: 0.894 ± 0.015
Exact Result: 1.0


In [35]:
# Plot comparing the different expectation values
plt.bar(
    ["Unmitigated", "TEM"],
    [unmitigated_evs, tem_evs],
    yerr=[unmitigated_stds, tem_std],
    color=["grey", "c"],
)
plt.hlines(y=exact_evs, xmin=-0.5, xmax=1.5, colors="r", linestyles="dashed")
plt.ylabel("Expectation Value")
plt.ylim(0, 1.1)
plt.show()

<Image src="../docs/images/tutorials/simulate-kicked-ising-tem/extracted-outputs/c3a2168d-98df-491e-a1f8-05de5684ab96-0.avif" alt="Output of the previous code cell" />

ジョブがプライベートに設定されていない場合、後で結果を取得できます。そのためには、ここで表示されるジョブIDを保存し、`tem_job_custom = catalog.get_job_by_id("your-job-id")` を使用します。

In [27]:
# Get the metadata of the TEM job
job_metadata = results_custom.metadata

# Get the runtime of the TEM job
qpu_runtime = job_metadata["resource_usage"]["RUNNING: EXECUTING_QPU"][
    "QPU_TIME"
]
classical_runtime = (
    job_metadata["resource_usage"]["RUNNING: OPTIMIZING_FOR_HARDWARE"][
        "CPU_TIME"
    ]
    + job_metadata["resource_usage"]["RUNNING: POST_PROCESSING"]["CPU_TIME"]
)

print(f"QPU Runtime: {qpu_runtime} seconds")
print(f"Classical Runtime: {classical_runtime} seconds")

QPU Runtime: 342.0 seconds
Classical Runtime: 107.632604 seconds


## Scale TEM to large circuits

Large circuits can, in principle, be run with the TEM function. However, it is important to be aware of the of the limitations of the classical resources, as TEM is executed on IBM Cloud runners with potentially very long run times. For extremely large circuits, contact the TEM support team at [qiskit_ibm@algorithmiq.fi](mailto:qiskit_ibm@algorithmiq.fi).

Here we run an example with a larger, utility-scale-sized 30-qubit circuit, optimizing the TEM parameters for speed rather than accuracy.

In [ ]:
# Kicked Ising model parameters
n_qubits = 30
t_steps = 15
h = 0.0

# Load the circuit for the kicked Ising model
circuit = load("ki_30q.qasm")


# Build the observable for the kicked Ising model
observable = xt_observable(n_qubits=n_qubits, t_steps=t_steps)

# Collect the input PUBs, in this case composed of a single circuit and observable
pubs = [(circuit, [observable])]

Let's define some performance-oriented options:

In [49]:
options = {
    "num_randomizations": 32,
    "max_layers_to_learn": 2,
    "shots_per_randomization": 128,
    "layer_pair_depths": [0, 1, 2, 4, 16, 32, 64],
    "default_shots": 5_000,
    "tem_max_bond_dimension": 128,
    "tem_compression_cutoff": 1e-10,
    "compute_shadows_bias_from_observable": True,
    "private": False,
}

Finally, run the experiment, get the result, and visualize it. This will take approximately 3.5 QPU minutes.

In [50]:
tem_job_large = tem.run(pubs=pubs, backend_name=backend_name, options=options)

In [51]:
job_id = tem_job_large.job_id
print(f"Job ID: {job_id}")

Job ID: 9f3f190f-f4b0-4dcb-bb83-5f71f37d0d77


In [53]:
results_large = tem_job_large.result()
tem_evs = results_large[0].data.evs[0]
tem_std = results_large[0].data.stds[0]

print(f"TEM Result: {tem_evs:.3f} ± {tem_std:.3f}")


# Get the metadata of the TEM job
job_metadata = tem_job_large.result().metadata

# Get the runtime of the TEM job
qpu_runtime = job_metadata["resource_usage"]["RUNNING: EXECUTING_QPU"][
    "QPU_TIME"
]
classical_runtime = (
    job_metadata["resource_usage"]["RUNNING: OPTIMIZING_FOR_HARDWARE"][
        "CPU_TIME"
    ]
    + job_metadata["resource_usage"]["RUNNING: POST_PROCESSING"]["CPU_TIME"]
)

print(f"QPU Runtime: {qpu_runtime} seconds")
print(f"Classical Runtime: {classical_runtime} seconds")

TEM Result: 0.794 ± 0.026
QPU Runtime: 203.0 seconds
Classical Runtime: 251.71805499999996 seconds


In [54]:
# Plot comparing the different expectation values
metadata_large = results_large[0].metadata
unmitigated_evs = metadata_large["evs_non_mitigated"][0]
unmitigated_stds = metadata_large["stds_non_mitigated"][0]

exact_evs = np.cos(2 * h) ** t_steps

plt.bar(
    ["Unmitigated", "TEM"],
    [unmitigated_evs, tem_evs],
    yerr=[unmitigated_stds, tem_std],
    color=["grey", "c"],
)
plt.hlines(y=exact_evs, xmin=-0.5, xmax=1.5, colors="r", linestyles="dashed")
plt.ylabel("Expectation Value")
plt.ylim(0, 1.1)
plt.show()

<Image src="../docs/images/tutorials/simulate-kicked-ising-tem/extracted-outputs/24894c44-e399-4b9d-a3ff-38a28ff32ece-0.avif" alt="Output of the previous code cell" />